In [1]:
from langchain_chroma import Chroma
import os
import chromadb
import pathlib
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

C:\Users\SURFACE\AppData\Local\Temp\ipykernel_15772\3558468604.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
# Source 1 - the PDF. PyPDFLoader returns one document per page.

pdf_docs = PyPDFLoader("victor_technical_report.pdf").load()

# Source 2 - the text file. Plain Python can read this on its own.
with open("business_registration_guide.txt", "r", encoding="utf-8") as f:
    txt_text = f.read()

print("Pages loaded from the PDF :", len(pdf_docs))
print("Characters read from TXT  :", len(txt_text))

print()
print("First 300 characters of the PDF:")
print(pdf_docs[0].page_content[:300])

print()
print("First 300 characters of the TXT:")
print(txt_text[:300])

Pages loaded from the PDF : 21
Characters read from TXT  : 4601

First 300 characters of the PDF:
TECHNICAL REPORT ON STUDENT INDUSTRIAL WORK 
EXPERIENCE SCHEME 
[SIWES] 
BY 
OGUNGBESAN VICTOR AYODEJI 
MATRICULATION NUMBER - 210504002 
DEPARTMENT OF MECHANICAL ENGINEERING 
FACULTY OF ENGINEERING 
EKITI STATE UNIVERSITY, ADO EKITI, EKITI STATE. 
AT 
PETIRABLE ELECTRIC POLES LTD 
KM 2 GBERIGBE – I

First 300 characters of the TXT:
GUIDE TO REGISTERING A BUSINESS IN NIGERIA
Corporate Affairs Commission (CAC)

WHY REGISTER

Registration with the Corporate Affairs Commission gives a business a legal
identity. Without it a business cannot open a corporate bank account, bid for
government or corporate contracts, obtain most catego


In [3]:
# The same splitter is used for both sources
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

# The PDF pages are already documents, so we split the documents
pdf_chunks = splitter.split_documents(pdf_docs)

# The TXT is one long plain string, so we create documents from it
txt_chunks = splitter.create_documents([txt_text])

print("Chunks from the PDF :", len(pdf_chunks))
print("Chunks from the TXT :", len(txt_chunks))


# ---------------------------------------------------------------
# CHOOSE WHICH SOURCE TO EMBED
# Uncomment the line you want and comment out the others.
# ---------------------------------------------------------------
chosen_chunks = pdf_chunks + txt_chunks     # use both together
# chosen_chunks = pdf_chunks                # use only the PDF
# chosen_chunks = txt_chunks                # use only the text file


# Chroma only needs the plain text of each chunk
documents = [chunk.page_content for chunk in chosen_chunks]

print()
print("Documents ready to embed:", len(documents))

Chunks from the PDF : 38
Chunks from the TXT : 7

Documents ready to embed: 45


In [4]:
#use this to comfirm the number documents to embed 
print(f"Total Documents to be embedded {len(documents)}\n")

Total Documents to be embedded 45



In [5]:
load_dotenv()

# ---------------------------------------------------------------
# PROVIDER CONFIG
# The API keys are secret, so they live in the .env file.
# Everything else is written out here so you can see it.
#
# NOTE: the chat model and the embedding model run on TWO DIFFERENT
# servers. Each one needs its own base URL and its own key.
# Do not mix them up.
# ---------------------------------------------------------------

# --- Chat model ---
api_key         = os.getenv("OPENAI_API_KEY")
base_url        = "https://museglimmer30b.publicaai.com/v1"
chat_model_name = "meta-models/Muse-Glimmer-30B"

# --- Embedding model (different server, different key) ---
embedding_api_key    = os.getenv("EMBEDDING_API_KEY")
embedding_base_url   = "https://qwen-embed.publicaai.com/v1"
embedding_model_name = "Qwen/Qwen3-Embedding-0.6B"

print("Chat base URL     :", base_url)
print("Chat model        :", chat_model_name)
print("Embed base URL    :", embedding_base_url)
print("Embedding model   :", embedding_model_name)

Chat base URL     : https://museglimmer30b.publicaai.com/v1
Chat model        : meta-models/Muse-Glimmer-30B
Embed base URL    : https://qwen-embed.publicaai.com/v1
Embedding model   : Qwen/Qwen3-Embedding-0.6B


In [6]:
chroma_path = "siwes_report_store"


In [7]:
# Initialize the embedding model
embeddings = OpenAIEmbeddings(
    model=embedding_model_name,
    api_key=embedding_api_key,
    base_url=embedding_base_url

)

In [8]:
# Initialize ChromaDB
siwes_report_vdb = Chroma(
    collection_name="siwes_report",
    embedding_function=embeddings,
    persist_directory=chroma_path
)


In [9]:
# Add documents to the vector database
siwes_report_vdb.add_texts(
    texts=documents
)

['610c4acf-6ba9-432a-8cf0-198060d097bb',
 '1c8f4478-5835-4189-a7c9-1d96f894c21d',
 'da787274-93f4-439f-9b7f-10ffccb5af84',
 'c7e6574b-ff0f-4dd7-a992-0a6c95e04092',
 'f81bf365-e5f1-42f3-bfb4-43aa84e2e4e4',
 'c04fecc7-a279-4213-ba7c-61005dd6895e',
 '92e05083-20f9-4754-a8bb-750d6faf7c31',
 '567b21f9-9a2f-42e7-b213-7a6be57bfcf4',
 'e1b42533-7609-422e-9e1e-67ab7390a810',
 'c790cd7b-f998-4140-bc56-b3f098055d95',
 '7d6743e2-021c-4ac5-8b35-07172627e852',
 'bfe4d6e6-b814-4189-a546-27c102f2681f',
 'e5f84418-4808-4c0a-9405-b2be0deeaf92',
 '2e3ef454-5a4b-4452-b4dd-c5dff49ade8d',
 '6ffe8b60-b695-4879-88d5-b6c3cf081f19',
 'e1f52d5e-7458-40fe-8945-192269c9a26a',
 '8d40a2d8-d7ce-43fe-a025-5ee676ea8ebe',
 '972388b8-979b-4b83-a67b-e69ff8055c6f',
 '24128543-c042-4051-bf8b-5df65fdcb57f',
 'e59c94f4-c810-4bac-b30e-93c7086cb0ef',
 '5ace7b30-70a7-4c6c-aaf5-aa2a9189296e',
 'acd95505-68ff-4756-8388-af5f408e2852',
 '99b94aeb-b922-47fb-ba28-819125713adf',
 '5126ce24-00c5-4757-9e30-22bcb8c43213',
 '60f113db-4362-

In [10]:
# Maximal Marginal Relevance (MMR) for diverse and relevant results.
siwes_report_retriever = siwes_report_vdb.as_retriever(search_type="mmr")

In [13]:
msme_retriever = siwes_report_vdb.as_retriever(search_type="mmr", search_kwargs={'k': 2, 'fetch_k': 10})

In [14]:
question = "What is the name of the company where siwes was undertaken?"
siwes_report_retriever.invoke(question)             

[Document(id='972388b8-979b-4b83-a67b-e69ff8055c6f', metadata={}, page_content='5 \n \n2.3 PETIRABLE SERVICES \nPertirable offers 3 distinct services that has served over 200 clients which includes \nindividuals and communities, government agencies and multinational corporations  such \nas : Ikeja Electric, MTN, IBEDC. These services includes: \na. Petirable Electric Poles Services \nPetirable is dedicated to the production of world -class concrete electric poles. The \norganisation specialises in producing durable, high-quality, and standard concrete electric \npoles designed to meet national and international standards. Their poles, which are SON \n(Standard Organisation of Nigeria) certified  are engineered to withstand Nigeria’s and \nAfrica’s diverse environmental conditions while ensuring safety and reliability in power \ndistribution. \nTheir range of products include:'),
 Document(id='c7e6574b-ff0f-4dd7-a992-0a6c95e04092', metadata={}, page_content='iii \n \nDEDICATION \nI dedi

In [15]:
#initialize the chatmodel
chatmodel = ChatOpenAI(
    api_key=api_key,
    base_url=base_url,
    model=chat_model_name,
    temperature=0
)

In [16]:
#using from template method
prompt = ChatPromptTemplate.from_template(
    """You are a student who just completed the student industrial working experience (SIWES) training and serves as a consultant providing insights on siwes training in Nigeria.
You will be provided with the context: {context} to answer the user's question.
The context includes sections about the company where the SIWES was carried out, the activities carried out during the experience, the lessons and tools used in carrying out activities during the training.
Provide a comprehensive response.
Include relevant sources or links from the context in your response at the end of each answer, include a statement: "To read more, check out this link: [insert link]."
Avoid unnecessary or unrelated details. Format the text output clearly and professionally in an HTML format.
question: {question}""")


In [17]:
question = "What is the name of the company where siwes was undertaken?"

#retrieve the document
get_doc = siwes_report_retriever.invoke(question)
# Prepare the input for the chain
input = {"context": get_doc, "question": question}

# Create the chain
chain = prompt | chatmodel | StrOutputParser()

answer = chain.invoke(input)
print(answer)

Based on the SIWES training report context provided, the industrial attachment was undertaken at:

**Petirable**

The report describes the company’s operations and services during the SIWES period:

* Petirable offers 3 distinct services that has served over 200 clients which includes individuals and communities, government agencies and multinational corporations such as : Ikeja Electric, MTN, IBEDC. These services includes:
* Petirable Electric Poles Services
Petirable is dedicated to the production of world -class concrete electric poles. The organisation specialises in producing durable, high-quality, and standard concrete electric poles designed to meet national and international standards. Their poles, which are SON (Standard Organisation of Nigeria) certified are engineered to withstand Nigeria’s and Africa’s diverse environmental conditions while ensuring safety and reliability in power distribution. [Document(id='972388b8-979b-4b83-a67b-e69ff8055c6f')]

To read more, check out 